In [13]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from collections import defaultdict
from itertools import cycle

# ML / NLP
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances
from bertopic import BERTopic

from sklearn.manifold import TSNE

In [55]:
# --- CONFIGURATION ---
# (Adjust these paths as needed)
ROOT_PATH = Path("/Users/cristian/Library/CloudStorage/GoogleDrive-cristianmejia00@gmail.com/My Drive/Bibliometrics_Drive") # Or local path
PROJECT_FOLDER = 'Q339_igem'
ANALYSIS_ID = 'a01_tm__f01_e01__hdbs'
SETTINGS_FILE = "settings_analysis_directive_2025-08-04-16-40.json"
LEVEL = '0'

In [56]:
# --- HELPER FUNCTIONS ---

def load_settings(path):
    with open(path, 'r') as f:
        return json.load(f)

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

In [57]:
def _compute_point_sizes_with_fallback(df_plot, primary_col='citations', fallback_col='degree', min_size=4.0, max_size=40.0):
    """
    Compute marker sizes from primary_col, and safely fall back to fallback_col if needed.
    """
    n_points = len(df_plot)
    if n_points == 0:
        return np.array([])

    if primary_col in df_plot.columns:
        values = pd.to_numeric(df_plot[primary_col], errors='coerce').fillna(0).clip(lower=0)
    else:
        values = pd.Series(np.zeros(n_points), index=df_plot.index)

    # Fallback when primary values are not informative (all equal)
    if values.nunique(dropna=True) <= 1 and fallback_col in df_plot.columns:
        fallback_values = pd.to_numeric(df_plot[fallback_col], errors='coerce').fillna(0).clip(lower=0)
        if fallback_values.nunique(dropna=True) > 1:
            values = fallback_values

    max_value = values.max()
    if max_value > 0:
        return min_size + (max_size - min_size) * (np.log1p(values) / np.log1p(max_value))

    return np.full(n_points, min_size)


def plot_clusters_improved(df_clean, rcs, output_path):
    """
    Plot clusters using df_clean which already contains x, y coordinates and colors.
    Cluster names are retrieved from rcs dataframe.
    """
    # Filter out rows without coordinates
    valid_mask = df_clean['x'].notna() & df_clean['y'].notna()
    df_plot = df_clean[valid_mask].copy()
    
    unique_clusters = sorted(df_plot['cluster_code'].unique())
    
    # Create cluster name mapping from rcs
    cluster_name_map = dict(zip(rcs['cluster_code'], rcs['cluster_label']))
    
    fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
    
    # Scale marker size by citations, with safe fallback to degree
    point_sizes = _compute_point_sizes_with_fallback(df_plot, primary_col='citations', fallback_col='degree')
    ax.scatter(df_plot['x'], df_plot['y'], 
               c=df_plot['color'], s=point_sizes, alpha=0.5, edgecolors='none')
    
    # Add Text Labels
    for cluster_code in unique_clusters:
        cluster_mask = df_plot['cluster_code'] == cluster_code
        
        # Get the median center of the cluster points
        x_center = df_plot.loc[cluster_mask, 'x'].median()
        y_center = df_plot.loc[cluster_mask, 'y'].median()
        
        # Get cluster name from rcs
        cluster_name = cluster_name_map.get(cluster_code, str(cluster_code))
        
        # Clean name logic
        words = str(cluster_name).split('_')
        short_name = " ".join(words[:3]) if len(words) > 2 else words[0]
        
        # Only label if the cluster is somewhat large to avoid clutter
        if sum(cluster_mask) > 20: 
            ax.text(x_center, y_center, short_name, fontsize=5, fontweight='bold', 
                    ha='center', va='center', alpha=0.9, color='black',
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.6, ec="none"))

    ax.axis('off')
    plt.tight_layout()
    fig.savefig(output_path / "plot_clusters_tsne.png")
    plt.close()

def plot_years_improved(df_clean, output_path):
    """
    Plot years using df_clean which already contains x, y coordinates and year.
    """
    # Filter out rows without coordinates
    valid_mask = (df_clean['x'].notna() & df_clean['y'].notna() & 
                  df_clean['year'].notna() & (df_clean['year'] > 1900))
    df_plot = df_clean[valid_mask].copy()
    
    # Sort by year so new papers are plotted ON TOP of old ones
    df_plot = df_plot.sort_values('year')

    fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
    
    # Scale marker size by citations, with safe fallback to degree
    point_sizes = _compute_point_sizes_with_fallback(df_plot, primary_col='citations', fallback_col='degree')

    # Using 'turbo' for a more "scientific" and impactful look
    sc = ax.scatter(df_plot['x'], df_plot['y'], 
                    c=df_plot['year'], cmap='turbo', 
                    s=point_sizes, alpha=0.5, edgecolors='none')
    
    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label('Year', rotation=270, labelpad=15)
    
    ax.axis('off')
    plt.tight_layout()
    fig.savefig(output_path / "plot_years_tsne.png")
    plt.close()

In [58]:
# 1. SETUP
base_dir = ROOT_PATH / PROJECT_FOLDER
settings_path = base_dir / ANALYSIS_ID / SETTINGS_FILE
settings = load_settings(settings_path)

In [59]:
# Load RCS
rcs = pd.read_csv(base_dir / ANALYSIS_ID / f'level{LEVEL}' / 'rcs_merged.csv').reset_index(drop=True)

# Define color palette
fukan_colors = ["#f00f15", "#2270e7", "#e5e510", "#ff8103", "#4f3dd1", "#26cc3a", "#ec058e", "#9cb8c2", "#fffdd0", "#b40e68"]
fukan_colors_extended = fukan_colors + ["#5afb5a", "#beaed4", "#fdc086", "#99fdff", "#c430ff", "#e4dbe0", "#bf5b17", "#666666"]

# Add color column to rcs if it doesn't exist
if 'color' not in rcs.columns:
    # Get the number of rows in rcs (equivalent to length(id_com) in R)
    n_rows = len(rcs)
    # Recycle color_palette to match the length of rcs
    import itertools
    color_palette = list(itertools.islice(itertools.cycle(fukan_colors_extended), n_rows))
    rcs['color'] = color_palette
    print(f"Added 'color' column to rcs with {n_rows} colors (recycled from {len(fukan_colors_extended)} base colors)")
else:
    print("'color' column already exists in rcs")

Added 'color' column to rcs with 69 colors (recycled from 18 base colors)


In [60]:
rcs.columns

Index(['cluster', 'cluster_code', 'main_cluster', 'cluster_name', 'documents',
       'documents_percent', 'documents_cummulative', 'PY_Min.', 'PY_X1st.Qu.',
       'PY_Median', 'PY_Mean', 'PY_X3rd.Qu.', 'PY_Max.', 'PY_sd', 'Z9_Min.',
       'Z9_X1st.Qu.', 'Z9_Median', 'Z9_Mean', 'Z9_X3rd.Qu.', 'Z9_Max.',
       'Z9_sd', 'participation', 'growth_rate', 'rcs_label', 'hub_title',
       'hub_year', 'hub_degree', 'hub_id', 'hub_type1', 'hub_type2', 'AU',
       'WC', 'Countries', 'Institutions', 'DE', 'ID', 'color'],
      dtype='object')

In [61]:
rcs['cluster_label'] = rcs['cluster_code'].astype(str) + "-" + rcs['hub_title']
rcs.head()

,cluster,cluster_code,main_cluster,cluster_name,documents,documents_percent,documents_cummulative,PY_Min.,PY_X1st.Qu.,PY_Median,...,hub_type1,hub_type2,AU,WC,Countries,Institutions,DE,ID,color,cluster_label
0,1,1,1,NaN,164,3.61,3.61,2009,2017.0,2020.5,...,ARTICLE,ARTICLE,nju-china; afcm-egypt; lzu-china; sysu-china; ...,"biofilms; vaccines; fertility, contraceptives,...",chn; usa; deu; grc; egy,"nanjing university\n\n\nnanjing, jiangsu, chin...",cancer; system; therapy; ; targeting,therapeutics; oncology; health & medicine; hig...,#f00f15,"1-A soothing nasal suction complex, no taboos ..."
1,2,2,2,NaN,129,2.84,6.45,2009,2014.0,2018.0,...,ARTICLE,ARTICLE,austin_utexas; yale; cbnu-korea; go_paris-sacl...,advancements in dna assembly; biosafety concer...,usa; chn; deu; can; fra,"tel aviv university \n\n\ntel aviv, israel\n\n...",synthetic; engineering; system; ; coli,foundational advance; new application; softwar...,#2270e7,2-Using synthetic biology to increase recombin...
2,3,3,3,NaN,129,2.84,9.29,2011,2018.0,2021.0,...,ARTICLE,ARTICLE,szu-china; tec-chihuahua; lethbridge; alma; co...,biological fertilizers; crop yields; diseases ...,chn; usa; can; mex; ind,"shenzhen university\n\n\nshenzhen city ,p.r.ch...",detection; rna; system; ; aflatoxin,agriculture; food & nutrition; environment; hi...,#e5e510,3-Defending North American Bats Against White ...
3,4,4,4,NaN,121,2.66,11.95,2010,2017.0,2019.0,...,ARTICLE,ARTICLE,bit; cmuq; exeter; grenoble-alpes; moscow,public health and pandemic preparedness; crisp...,chn; usa; gbr; can; deu,"university of exeter; acibadem university, not...",detection; diagnostic; rapid; system;,diagnostics; high school; health & medicine; n...,#ff8103,4-Behcheck
4,5,5,5,NaN,121,2.66,14.61,2009,2015.0,2019.0,...,ARTICLE,ARTICLE,fdr-hb_peru; hust-china; cornell; epfl; hong_k...,heavy metals; household and industrial waste; ...,chn; usa; can; deu; gbr,"university of lethbridge\n\n\nlethbridge, albe...",metal; heavy; cadmium; bioremediation; system,environment; high school; bioremediation; new ...,#4f3dd1,5-Enhanced Vitamin B12 Synthesis from the Wast...


In [62]:
# Load coords
coords = pd.read_csv(base_dir / ANALYSIS_ID/ 'document_coords_tsne.csv')

# Rename columns to x and y
coords.columns = ['uuid', 'UT', 'x', 'y']

coords.head()

,uuid,UT,x,y
0,f6c17ca4-e76a-4c53-80cd-349fe34b38d5,173,33.336258,65.509460
1,f7173d50-7003-491e-a6ff-527fc1055e00,174,-76.321360,-46.922127
2,8169f0c0-96ec-4ee5-9458-bb03b9f1ef28,175,5.412798,-13.836423
3,f0c10215-5236-419b-8112-635791483c0a,176,27.927290,-9.688766
4,9b1cd353-7137-46d7-bdc5-be1f7d4694e2,177,81.770110,7.711818


In [63]:
# Load article report
#article_report = pd.read_csv(base_dir / ANALYSIS_ID / f'level{LEVEL}' / 'article_report.csv', encoding='cp1250').reset_index(drop=True)
article_report = pd.read_csv(base_dir / ANALYSIS_ID / f'level{LEVEL}' / 'article_report.csv').reset_index(drop=True)
article_report.head()

,Cluster Index,Cluster Code,Authors,Publication Years,DOI,Title,Abstract,Citations,Degree,Author Keywords,Categories,Countries,ID,uuid
0,1,1,Squirrel-Beijing-I,2024,https://doi.org/https://2024.igem.wiki/squirre...,"A soothing nasal suction complex, no taboos to...",Riding a roller coaster is a thrilling enterta...,1,1.000000,; soothing; nasal; suction; complex; taboos; r...,RNAi/siRNA viral vectors; Biofilms; Gene thera...,CHN,5081,c479519a-ef6c-4a63-a3bb-7a0051484db3
1,1,1,SubCat_Shanghai,2022,https://doi.org/https://2022.igem.wiki/subcat-...,Say goodbye to tumor resistance,Glioma is the most common malignant tumor of t...,1,0.875921,say; goodbye; tumor; resistance,NaN,CHN,4251,fa8e9751-4d0d-43e1-92eb-1e7f25c9d68d
2,1,1,XJTLU-CHINA,2019,https://doi.org/https://2019.igem.org/Team:XJT...,exoCar in the brain,exoCar is designed to apply mRNA-contained exo...,1,0.799751,exocar; brain,NaN,CHN,3030,bd9de715-ba27-4fab-b287-ceea54aedece
3,1,1,Linkoping,2020,https://doi.org/https://2020.igem.org/Team:Lin...,ClusteRsy: A user-friendly software for transc...,Asthma is a chronic and inflammatory disease t...,1,0.756368,clustersy; userfriendly; software; transcripto...,NaN,SWE,3647,66a28d9e-226b-4f5b-b80f-f5694b74ef54
4,1,1,TJI-Seoul,2024,https://doi.org/https://2024.igem.wiki/tji-seoul,Image-Based Machine Learning of Keratin-Interm...,This project aims to develop an innovative mac...,1,0.727348,imagebased; machine; learning; keratinintermed...,Immune-cell-based cancer therapies; Re-program...,KOR,5421,8dab33dd-7ae9-4ebb-8432-44b4e8f298e7


In [64]:
# Create df_clean
# 1. Select and rename columns from article_report
df_clean = article_report[['uuid', 'ID', 'Publication Years', 'Citations', 'Degree', 'Cluster Code']].copy()
df_clean.columns = ['uuid', 'UT', 'year', 'citations', 'degree', 'cluster_code']

# Fix uuid encoding: ensure both are strings and strip whitespace
df_clean['uuid'] = df_clean['uuid'].astype(str).str.strip()
coords_fixed = coords.copy()
coords_fixed['uuid'] = coords_fixed['uuid'].astype(str).str.strip()

# 2. Merge with coords to add x and y columns
df_clean = df_clean.merge(coords_fixed, on='UT', how='left')

# 3. Assign colors by matching cluster with rcs
# Create a mapping from cluster to color from rcs
cluster_color_map = dict(zip(rcs['cluster_code'], rcs['color']))
df_clean['color'] = df_clean['cluster_code'].map(cluster_color_map)

print(f"df_clean shape: {df_clean.shape}")
#print(f"Rows with coordinates: {df_clean['x'].notna().sum()}")
df_clean.head()

df_clean shape: (4548, 10)


,uuid_x,UT,year,citations,degree,cluster_code,uuid_y,x,y,color
0,c479519a-ef6c-4a63-a3bb-7a0051484db3,5081,2024,1,1.000000,1,c479519a-ef6c-4a63-a3bb-7a0051484db3,93.828690,-28.662909,#f00f15
1,fa8e9751-4d0d-43e1-92eb-1e7f25c9d68d,4251,2022,1,0.875921,1,fa8e9751-4d0d-43e1-92eb-1e7f25c9d68d,91.578354,-33.163784,#f00f15
2,bd9de715-ba27-4fab-b287-ceea54aedece,3030,2019,1,0.799751,1,bd9de715-ba27-4fab-b287-ceea54aedece,90.922640,-24.698109,#f00f15
3,66a28d9e-226b-4f5b-b80f-f5694b74ef54,3647,2020,1,0.756368,1,66a28d9e-226b-4f5b-b80f-f5694b74ef54,81.282585,-33.188328,#f00f15
4,8dab33dd-7ae9-4ebb-8432-44b4e8f298e7,5421,2024,1,0.727348,1,8dab33dd-7ae9-4ebb-8432-44b4e8f298e7,81.498825,-43.988100,#f00f15


## Visualize with TSNE

In [65]:
# Pass the t-SNE coordinates to your plotting functions
# Note: I slightly increased point size (s=1.5) and alpha (0.5) for better visibility
output_dir = base_dir / ANALYSIS_ID / f'level{LEVEL}'

plot_clusters_improved(df_clean, rcs, output_dir)

plot_years_improved(df_clean, output_dir)

print("Pipeline Finished Successfully.")

Pipeline Finished Successfully.


---

# Centroid-based plots
Instead of plotting individual data points, we can visualize the centroids of clusters to get a clearer overview of the data distribution. The size of each centroid represents the number of points in that cluster.

In [66]:
def plot_centroid_bubbles(df_clean, rcs, output_path):
    """
    Plots 1 bubble per cluster at the centroid location using df_clean and rcs.
    Size = Number of papers.
    Color = From rcs dataframe.
    """
    
    # 1. Setup Data - Filter out rows without coordinates
    valid_mask = df_clean['x'].notna() & df_clean['y'].notna()
    df_plot = df_clean[valid_mask].copy()
    
    unique_clusters = sorted(df_plot['cluster_code'].unique())
    
    # 2. Create mappings from rcs
    cluster_color_map = dict(zip(rcs['cluster_code'], rcs['color']))
    cluster_name_map = dict(zip(rcs['cluster_code'], rcs['cluster_label']))
    
    # 3. Aggregate Data
    centroids = []
    sizes = []
    colors = []
    labels = []
    
    for cluster_code in unique_clusters:
        cluster_mask = df_plot['cluster_code'] == cluster_code
        
        # Location: Mean of all points in this cluster
        x_center = df_plot.loc[cluster_mask, 'x'].mean()
        y_center = df_plot.loc[cluster_mask, 'y'].mean()
        centroids.append([x_center, y_center])
        
        # Size: Number of papers
        count = sum(cluster_mask)
        sizes.append(count)
        
        # Color from rcs
        colors.append(cluster_color_map.get(cluster_code, '#666666'))
        
        # Label from rcs
        cluster_name = cluster_name_map.get(cluster_code, str(cluster_code))
        # Clean: "10_dna_rna_gene" -> "dna rna gene"
        words = str(cluster_name).split('_')
        short_name = " ".join(words[:3]) if len(words) > 2 else words[0]
        labels.append(short_name)
        
    centroids = np.array(centroids)
    sizes = np.array(sizes)
    
    # 4. Scaling the Bubble Size
    # We normalize size so the largest bubble is ~5000 points area
    # and the smallest is visible.
    # Adjust 'scale_factor' to make bubbles bigger/smaller overall.
    scale_factor = 5000 
    norm_sizes = (sizes / sizes.max()) * scale_factor
    # Ensure minimum visibility for tiny clusters
    norm_sizes = np.maximum(norm_sizes, 50) 
    
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
    
    # Scatter with transparency and white edges for "Bubble" look
    ax.scatter(centroids[:, 0], centroids[:, 1], 
               s=norm_sizes, c=colors, alpha=0.7, 
               edgecolors='white', linewidth=1.5)
    
    # 6. Labeling
    # We create a hierarchy: labels on huge bubbles are big, small are small
    for i, txt in enumerate(labels):
        # Calculate appropriate font size based on bubble size
        # Heuristic: 6pt minimum, 12pt maximum
        f_size = 6 + (norm_sizes[i] / norm_sizes.max()) * 8
        
        # Only label if cluster is significant enough to not clutter
        if sizes[i] > (sizes.max() * 0.02): 
            ax.text(centroids[i, 0], centroids[i, 1], txt, 
                    ha='center', va='center', 
                    fontsize=f_size, fontweight='bold', color='black',
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.3, ec="none"))

    ax.axis('off')
    plt.tight_layout()
    fig.savefig(output_path / "plot_centroids_sized.png")
    print(f"Saved centroid bubble plot to {output_path}")
    plt.close()

In [67]:
    
# 3. Centroid Bubbles
plot_centroid_bubbles(df_clean, rcs, output_dir)

print("Pipeline Finished Successfully.")

Saved centroid bubble plot to /Users/cristian/Library/CloudStorage/GoogleDrive-cristianmejia00@gmail.com/My Drive/Bibliometrics_Drive/Q339_igem/a01_tm__f01_e01__hdbs/level0
Pipeline Finished Successfully.
